In [ ]:
import numpy as np
import os
from cloudvolume import CloudVolume, Skeleton
import logging
import glob
import json
import neuroglancer
import requests
import tifffile

In [ ]:
def add_point_annot(viewer, annot_df, point_trans=[0,0,0]):
    with viewer.txn() as s:
        #define an annotation layer
        s.layers['annotation'+str(i)] = neuroglancer.AnnotationLayer()
        annotations = s.layers['annotation'+str(i)].annotations

        #iterate over csv and add annotations
        counter = 1
        for index,row in annot_df.iterrows():
            pt = neuroglancer.PointAnnotation(id=str(counter), point=[row['x'],row['y'],row['z']])
            annotations.append(pt)
            counter += 1
            
    print(viewer)

def add_line_annot(viewer, annot_df, point_trans=[0,0,0]):
    with viewer.txn() as s:
        #define an annotation layer
        s.layers['annotation'+str(i)] = neuroglancer.AnnotationLayer()
        annotations = s.layers['annotation'+str(i)].annotations

        #iterate over csv and add annotations
        counter = 1
        for index,row in annot_df.iterrows():
            pt = neuroglancer.PointAnnotation(point_a=[row['x1'],row['y1'],row['z1']],point_b=[row['x2'],row['y2'],row['z2']], id=str(counter))
            annotations.append(pt)
            counter += 1

    print(viewer)

def zarr_to_ngl_viewer(zarr_fpath, ip = 'localhost', port='9999', server='http://bigkahuna.corp.alleninstitute.org', aff_transf=[0,0,0,0,0,0], im_pix_range=[0,10000], im_opacity=0.5):
    neuroglancer.set_server_bind_address(bind_address=ip,bind_port=port)
    viewer=neuroglancer.Viewer()

    zarr_attr = zarr_fpath + '/.zattrs'
    f = open(zarr_attr)
    data = json.load(f)

    # Alter dimension order and set dimension scale
    scale = np.array(data['multiscales'][0]['datasets'][0]['coordinateTransformations'][0]['scale'][2:5])
    dim_im = neuroglancer.CoordinateSpace(
                names=['z', 'y', 'x', 't'],
                units='nm',
                scales=scale,
            )

    # Create coordinate transform for image
    x1,x2,y1,y2,z1,z2 = aff_transf
    im_matrix = np.array([[1,x1,x2,0],[y1,1,y2,0],[z1,z2,1,0]])
    tr_im = neuroglancer.CoordinateSpaceTransform(input_dimensions = dim_im, output_dimensions =  dim_im, matrix = im_matrix) 
    
    with viewer.txn() as s:
        s.dimensions = dim_im
        s.layers['image'] = neuroglancer.ImageLayer(source=['zarr://' + server + zarr_fpath])
        s.layers['image'].layer.source[0].transform  = tr_im
        s.layers['image'].layer.shaderControls = {'normalized': {'range': im_pix_range}}
        s.layers['image'].layer.opacity = im_opacity

    return viewer

In [ ]:
zarr_to_ngl_viewer('/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S32_230412_highres/H17_x55_S32_230412_highres.zarr/highres_Pos105/', ip = 'localhost', port='9999')

In [ ]:
def generate_ngl_tiffs(source_path, out_path, chunk_size, resolution):
    """Create an neuroglancer precomputed volume using tiff files.
       source_path: directory underwhich tiff files will be found.
       out_path: directory for the precomputed volume.
    """
    
    # Read tiffstack into memory
    files = glob.glob(f"{source_path}/*.tif")
    files = sorted(files)
    data = None

    for layer in range(len(files)):
        image = tifffile.imread(files[layer])
        # Allocate array if needed; use the number of files and dimensions of first file
        if data is None:
            data = np.zeros(shape=(image.shape[1], image.shape[0], len(files)), dtype=image.dtype)
        data[:, :, layer] = image.T   # Tiff files have X & Y swapped

    info = CloudVolume.create_new_info(
        num_channels    = 1,
        layer_type      = 'image',
        data_type       = 'uint8', # Channel images might be 'uint8'
        # raw, png, jpeg, compressed_segmentation, fpzip, kempressed, compresso
        encoding        = 'png', 
        resolution      = resolution, # Voxel scaling, units are in nanometers
        voxel_offset    = [0, 0, 0], # x,y,z offset in voxels from the origin
        chunk_size      = chunk_size, # units are voxels
        volume_size     = data.shape # e.g. a cubic millimeter dataset
        )

    vol = CloudVolume(f'file://{out_path}', info=info, compress='', cache=False)
    logging.info(f"Creating cloud volume: {vol.info}")
    vol.commit_info()
    vol.commit_provenance()
    vol[:,:,:] = data.astype(np.uint8)

def generate_ngl_segmentation_empty(out_path):
    """Create an empty neuroglancer precomputed segmentation volume.
       out_path: directory where new the new cloud volume segmentation layer should be generated
    """
    
    info = CloudVolume.create_new_info(
        num_channels    = 1,
        layer_type      = 'segmentation',
        data_type       = 'uint64', # Channel images might be 'uint8'
        # raw, png, jpeg, compressed_segmentation, fpzip, kempressed, compresso
        encoding        = 'compressed_segmentation', 
        resolution      = [0,0,0], # Voxel scaling, units are in nanometers
        voxel_offset    = [0, 0, 0], # x,y,z offset in voxels from the origin
        chunk_size      = [0,0,0], # units are voxels
        volume_size     = [0,0,0], # e.g. a cubic millimeter dataset
        skeletons       = 'skeletons'
        )

    vol = CloudVolume(f'file://{out_path}', info=info, compress='', cache=False)
    logging.info(f"Creating cloud volume: {vol.info}")
    vol.commit_info()
    vol.commit_provenance()
        
        
def generate_ngl_skeletons(source_path, out_path, match_fname=False):
    """Generate skeletons from SWC files.
       This currently assumes the neuroglancer precomputed volume has already been generated by generate_ngl_segmentation.
       source_path: directory underwhich `swc_files_nm` will be found.
       out_path: directory with the previously generated segmentation layer.
    """

    vol = CloudVolume(f'file://{out_path}', compress='')
    vol.skeleton.meta.info.pop("vertex_attributes", None)
    vol.skeleton.meta.commit_info()

    files = glob.glob(f"{source_path}/*.swc")
    files = sorted(files)
    skel_dir = os.path.join(out_path, "skeletons")
    if not os.path.exists(skel_dir):
        os.makedirs(skel_dir)
    
    segprops = {"@type": "neuroglancer_segment_properties",
            "inline" : {
                "ids" : [],
                "properties" : [
                    {"id": "tags",
                        "type": "tags",
                        "tags" : ["all"],
                        "values" : []
                    },
                    {"id": "length",
                        "type": "number",
                        "data_type" : "float32",
                        "values" : []
                    }
                ]},
            }
    
    sids = list(range(len(files)))
    for ind,filename in enumerate(files):
        # ..../NNNN.swc -> NNNN
        with open(filename, mode='r') as f:
            swc = f.read()
        skel = Skeleton.from_swc(swc)

        if match_fname==True:
            sid = filename.split(".swc")[0].split("/")[-1]
            skel.id = sid
            
        else:
            sid = sids[ind]
            skel.id = sid 
 
        vol.skeleton.upload(skel)

        segprops["inline"]["ids"].append(str(sid))
        segprops["inline"]["properties"][0]["values"].append([0])  # tags
        segprops["inline"]["properties"][1]["values"].append(str(skel.cable_length()))
            
    # Write the segment properties
    segment_info_dir = os.path.join(out_path, "skeletons/segment_properties")
    os.makedirs(segment_info_dir, exist_ok=True)
    with open(os.path.join(segment_info_dir, "info"), "w") as f:
        json.dump(segprops, f)
        
    # Re-write info file with added segment_properties
    with open(f'{os.path.join(out_path, "skeletons")}/info', 'r') as f:
        infofile = json.load(f)
    infofile['segment_properties'] = 'segment_properties'
    
    with open(f'{out_path}skeletons/info', 'w') as f:
        json.dump(infofile, f)

def ngl_load_ImageWITHSkel(viewer, zarr_path, skel_path, skel_mip, skel_transl=[0,0,0], aff_transf=[0,0,0,0,0,0], pix_range=[0,10000]):
    """Load a single image zarr and its associated skeleton data into an existing neuroglancer viewer instance.
       zarr_path: directory underwhich the zarr image data can be found.
       skel_path: directory underwhich the precomputed volume for skeletons can be found.
       skel_mip: mip level of the image data used to generate the skeletons
       NOTE: skeletons assumed to have their z and x dimensions reversed
    """
    
    zarr_attr = zarr_path + '/.zattrs'
    f = open(zarr_attr)
    data = json.load(f)
    
    # Pull translations and scales for image and skeletons
    transl_im = np.array(data['multiscales'][0]['coordinateTransformations'][0]['translation'])[2:5]
    scale_im = np.array(data['multiscales'][0]['datasets'][0]['coordinateTransformations'][0]['scale'][2:5])
    transl_skel = np.array([transl_im[0]/scale_im[0],transl_im[1]/scale_im[1],transl_im[2]/scale_im[2]]).astype(float)
    scale_skel = np.array(data['multiscales'][0]['datasets'][skel_mip]['coordinateTransformations'][0]['scale'][2:5])

    # Alter dimension order and set dimension scale
    dim_im = neuroglancer.CoordinateSpace(
                names=['z', 'y', 'x', 't'],
                units='nm',
                scales=scale_im*1000,
            )

    dim_skel = neuroglancer.CoordinateSpace(
                names=['z', 'y', 'x', 't'],
                units='nm',
                scales=scale_skel*1000,
            )

    # Create coordinate transforms to adjust for zarr mip level and translations
    x1,x2,y1,y2,z1,z2 = aff_transf
    im_matrix = np.array([[1,x1,x2,0],[y1,1,y2,0],[z1,z2,1,0]])
    skel_matrix = np.array([[1,x1,x2,transl_skel[0]/(1+skel_mip) + skel_transl[0]],[y1,1,y2,transl_skel[1]/(1+skel_mip) + skel_transl[1]],[z1,z2,1,transl_skel[2]/(1+skel_mip) + skel_transl[2]]])
    tr_im = neuroglancer.CoordinateSpaceTransform(input_dimensions = dim_im, output_dimensions =  dim_im, matrix = im_matrix) 
    tr_skel = neuroglancer.CoordinateSpaceTransform(input_dimensions = dim_skel, output_dimensions = dim_skel, matrix = skel_matrix)
    
    # Load image and skeletons
    strip = zarr_path.split("_")[-1]
    with viewer.txn() as s:
        s.dimensions = dim_im
        s.layers['Image_' + strip] = neuroglancer.ImageLayer(source=['zarr://http://bigkahuna.corp.alleninstitute.org' + zarr_path])
        s.layers['Image_' + strip].layer.source[0].transform  = tr_im
        s.layers['Image_' + strip].layer.shaderControls = {'normalized': {'range': pix_range}}
        
    with viewer.txn() as s:
        s.layers['Skel_' + strip] = neuroglancer.SegmentationLayer(source=['precomputed://http://bigkahuna.corp.alleninstitute.org' + skel_path])
        s.layers['Skel_' + strip].layer.source[0].transform  = tr_skel

    return viewer


In [ ]:
def make_neuroglancer_url_vneurodata(state,
                                     base_url="http://bigkahuna.corp.alleninstitute.org/neuroglancer",
                                     state_url="https://json.neurodata.io/v1"):
    r = requests.post(state_url, json=state)
    json_url = r.json()["uri"]
    link = f"{base_url}/#!{json_url}"
    print(link)

def tifs_to_ngl_link(source_path, out_path, res, ip = 'localhost', port='9999'):
    """Create a neuroglancer viewer instance, create a precomputed volume using tiff files, then load the viewer.
       source_path: directory underwhich the tiff files will be found.
       out_path: directory where the precomputed volume will go.
    """
    
    neuroglancer.set_server_bind_address(bind_address=ip,bind_port=port)
    viewer=neuroglancer.Viewer()
    
    #generate precomputed volume for tif/tifs
    generate_ngl_tiffs(source_path, out_path, [512, 512, 64], res)
    
    with viewer.txn() as s:
        s.layers['Image'] = neuroglancer.ImageLayer(source=['precomputed://http://bigkahuna.corp.alleninstitute.org/' + out_path])
            
    view = s.to_json()
    make_neuroglancer_url_vneurodata(view)

def create_ngl_link_StripsWITHSkels(zarr_path, strip_range, skels_path, skels_mip, ip = 'localhost', port='9999', skels_transl=[0,0,0], aff_transf=[0,0,0,0,0,0], pix_range=[0,10000]):
    """Create a neuroglancer viewer instance, then load multiple image zarrs and their associated skeleton data into it.
       zarr_path: directory underwhich the zarr image data can be found.
       strip_range: the list range of strips being visualized (EX: [0,10])
       skels_path: directory underwhich the precomputed volume for skeletons can be found (with strip subdirectories).
       skels_mip: mip level of the image data used to generate the skeletons
    """
    
    neuroglancer.set_server_bind_address(bind_address=ip,bind_port=port)
    viewer=neuroglancer.Viewer()

    #Load image and skeleton data
    for strip in range(s_range[0],s_range[1]+1):
        impath= zarr_path +'highres_Pos' + str(strip)
        skpath= skels_path + 'Pos' + str(strip) + '/skeletons/'
        viewer = ngl_load_ImageWITHSkel(viewer, impath, skpath, skels_mip, skel_transl=skels_transl, aff_transf=aff_transf, pix_range=pix_range)

    #Create shareable neuroglancer link
    with viewer.txn() as s:
        view = s.to_json()
    make_neuroglancer_url_vneurodata(view)    

Create computed volumes for existing SWC skeletons, and load them into a neuroglancer viewer instance along with their associated zarr image files. They will will return a shareable neuroglancer link that you can use to view the data.

In [ ]:
###Create precomputed volumes for skeletons
s_range = [52,54]
indir = '/ACdata/Users/connorl/Skeletons/For_Kevin/S32_Pos52,53,54_MIP0/' ###skeletons
outdir = '/ACdata/Users/connorl/Neuroglancer/S32_Skeletons_Mip0/'

for n in range(s_range[0],s_range[1]+1):
    pos_dir = indir + 'Pos' + str(n) + "/"
    os.makedirs(outdir + 'Pos' + str(n) + "/", exist_ok=True)
    generate_ngl_segmentation_empty(outdir + 'Pos' + str(n) + "/")
    generate_ngl_skeletons(source_path=pos_dir + "Reconnected/reconnected_skeletons/" , out_path=outdir + 'Pos' + str(n) + "/")

In [ ]:
###Create neuroglancer link for multiple strips and their associated skeletons
s_range = [52,53]
skels_dir = '/ACdata/Users/connorl/Neuroglancer/S32_Skeletons_Mip0/'
outdir = '/ACdata/Users/connorl/Neuroglancer/S32_Skeletons_Mip0/'
zarr_dir = '/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S32_230412_highres/H17_x55_S32_230412_highres.zarr/'

create_ngl_link_StripsWITHSkels(zarr_path=zarr_dir, strip_range=s_range, skels_path=skels_dir, skels_mip=0, skels_transl=[23000,0,0], aff_transf=[0,0,0,0,0,0])

Set the source path for the folder with one or more tif files, and the out path for the precomputed layer files. The function will return a shareable neuroglancer link that you can use to view the data.

In [ ]:
tifs_to_ngl_link(source_path = '/ACdata/Users/connorl/stacks/',  out_path = '/ACdata/Users/connorl/stacks/precomputed/')